In [ ]:
#| default_exp loss

# Loss functions

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F, torch.nn as nn

In [ ]:
#| export
class FocalLoss(nn.Module):
    """
    adapted from tsai, weighted multiclass focal loss
    https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/losses.py#L116C1-L140C20
    """
    def __init__(self, 
                 weight=None, 
                 gamma=2., 
                 reduction='mean',
                 ignore_index=-100
                 ):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
        self.ignore_index = ignore_index
    
    __name__ = 'focalloss'
        
    def forward(self, x, y):
        """
        x: [bs x n classes x n patches]
        y: [bs x n patches]
        """
        log_prob = F.log_softmax(x, dim=1)
        prob = log_prob.exp()
        weight = self.weight.to(x.device) if self.weight is not None else None
        if y.dim() == 2:
            # hard labels
            ce = F.nll_loss(log_prob, y, weight=weight, reduction='none', ignore_index=self.ignore_index)
            loss = (1 - prob) ** self.gamma * ce.unsqueeze(1)
            mask = (y != self.ignore_index).float().unsqueeze(1)
        else:  # soft labels
            ce = -(y * log_prob)  # [bs x n_classes x n_patches]
            loss = (1 - prob) ** self.gamma * ce
            # Positions to ignore will have all zeros
            mask = (y.sum(dim=1) > 0).float().unsqueeze(1)
            
            if weight is not None:
                loss = loss * weight.view(1, -1, 1)
        if self.reduction == 'mean':
            loss = loss.sum() / mask.sum().clamp(min=1e-5)
        elif self.reduction == 'sum':
            loss = loss.sum()
        return loss

In [ ]:
#| notest
criterion = FocalLoss(gamma=0.7, weight=None, ignore_index=0)
batch_size = 10

n_patch = 721
n_class = 5
#m = torch.nn.Softmax(dim=-1)
logits = torch.randn(batch_size, n_class, n_patch)
target = torch.randint(0, n_class, size=(batch_size, n_patch))
criterion(logits, target)

tensor(8.4217)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()